# Zajęcia 1 — pierwszy eksperyment ML

**Wzorcowe rozwiązanie**

Notebook przedstawia kompletne wykonanie mikroinstrukcji z materiału do pierwszych zajęć. Każdy krok znajduje się w osobnej parze komórek: opis oraz implementacja. Notebook należy uruchamiać z katalogu głównego repozytorium.


# 0. Stworzenie środowiska i instalacja bibliotek

Ten krok wymaga posiadania wersji Python na komputerze. Mozna go zainstalować na wiele sposobów np. z tej [strony](https://www.python.org/downloads/).

**Wymagana wersja Pythona: 3.12** (np. 3.12.x). Część bibliotek z `requirements.txt` — w szczególności `nannyml` używany na spotkaniu 12 — nie działa jeszcze na Pythonie 3.13 ani nowszym. Jeśli w systemie zainstalowana jest tylko nowsza wersja, doinstaluj dodatkowo Pythona 3.12 i użyj go do stworzenia `.venv`, zamiast domyślnego `python`.

### macOS

1. Sprawdź, czy masz już Pythona 3.12: `python3.12 --version`. Jeśli komenda nie działa, zainstaluj go przez Homebrew: `brew install python@3.12` (jeśli nie masz Homebrew, zainstaluj go najpierw ze strony [brew.sh](https://brew.sh/)).
2. Otwórz terminal w głównym folderze projektu i utwórz środowisko wskazując konkretną wersję: `python3.12 -m venv .venv`.
3. Aktywuj środowisko: `source .venv/bin/activate`. Nazwa `.venv` powinna pojawić się na początku linii w terminalu.
4. Zainstaluj wymagane biblioteki: `pip install -r requirements.txt`.

### Windows

1. Sprawdź, czy masz już Pythona 3.12: `py -3.12 --version`. Jeśli komenda nie działa, zainstaluj Pythona 3.12 ze [strony python.org](https://www.python.org/downloads/) (podczas instalacji zaznacz opcję „Add python.exe to PATH”) lub przez Windows Installer z Microsoft Store.
2. Otwórz terminal (PowerShell) w głównym folderze projektu i utwórz środowisko wskazując konkretną wersję: `py -3.12 -m venv .venv`.
3. Aktywuj środowisko: `.\.venv\Scripts\Activate.ps1`. Jeśli PowerShell zgłosi błąd dotyczący polityki wykonywania skryptów, uruchom raz `Set-ExecutionPolicy -Scope CurrentUser RemoteSigned`, a następnie powtórz krok aktywacji.
4. Zainstaluj wymagane biblioteki: `pip install -r requirements.txt`.

Plik `requirements.txt` zawiera zależności dla **wszystkich** zajęć, więc instalację wykonujesz tylko raz. Po aktywacji środowiska w terminalu powinien być widoczny prefiks `(.venv)` — to znak, że biblioteki instalują się w prawidłowym, izolowanym środowisku projektu, a nie globalnie w systemie.

## 1. Import bibliotek

Importujemy wyłącznie elementy potrzebne do analizy danych, trenowania modeli, ewaluacji, wizualizacji i zapisu artefaktów.


In [ ]:
from pathlib import Path
import json

import joblib
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.datasets import load_iris
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import sys

repo_root = Path.cwd()
if not (repo_root / "skip_kernel_extension.py").exists():
    repo_root = repo_root.parent  # uruchomienie z katalogu notebooka
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

%load_ext skip_kernel_extension

## 2. Katalogi na artefakty

Tworzymy osobne katalogi na raporty i modele. `parents=True` tworzy także brakujące katalogi nadrzędne, a `exist_ok=True` pozwala bezpiecznie ponownie wykonać komórkę.


In [ ]:
ARTIFACTS_DIR = Path("artifacts/zajecia_01")
REPORTS_DIR = ARTIFACTS_DIR / "reports"
MODELS_DIR = ARTIFACTS_DIR / "models"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)


## 3. Wczytanie zbioru Iris

`load_iris(as_frame=True)` zwraca dane w formacie pandas i nie wymaga pobierania osobnego pliku z internetu. Rozdzielamy cechy `X` i etykietę `y`.


In [ ]:
iris = load_iris(as_frame=True)
X = iris.data.copy()
y = iris.target.copy()

print(iris.DESCR)


## 4. Pierwsze wiersze danych

Wyświetlamy próbkę, aby sprawdzić znaczenie kolumn i ogólny format danych.

**Zadanie:** w pierwszej komórce spróbuj samodzielnie wyświetlić kilka pierwszych wierszy `X`. Druga komórka zawiera rozwiązanie oznaczone `%%skip True` — domyślnie jest pomijane. Aby je zobaczyć w działaniu, zmień nagłówek na `%%skip False` (możesz też zwinąć jej kod, klikając strzałkę po lewej stronie komórki).


In [ ]:
%%skip True
# Rozwiązanie — zmień na `%%skip False`, aby wykonać tę komórkę.
display(X.head())


## 5. Liczba wierszy i kolumn

Sprawdzamy podstawowy rozmiar zbioru.


In [ ]:
# Sprawdź ile dataset ma wierszy i kolumn




In [ ]:
%%skip True
rows, columns = X.shape
print(f"Liczba wierszy: {rows}")
print(f"Liczba kolumn cech: {columns}")


## 6. Nazwy i typy kolumn

Typy danych wpływają na dostępne transformacje oraz sposób walidacji.


In [ ]:
# Sprawdź jakie są typy danych (dtypes)


In [ ]:
%%skip True
column_schema = pd.DataFrame({
    "column": X.columns,
    "dtype": X.dtypes.astype(str).values,
})
display(column_schema)


## 7. Brakujące wartości

Liczymy braki osobno dla każdej cechy.


In [ ]:
# Sprawdź ile brakujących wartości jest w każdej kolumnie


In [ ]:
%%skip True
missing_values = X.isna().sum().rename("missing_values")
display(missing_values.to_frame())


## 8. Zduplikowane wiersze

Duplikat nie zawsze jest błędem, ale jego liczba powinna być znana przed treningiem.


In [ ]:
# Sprawdź ile dataset ma zduplikowanych wierszy


In [ ]:
%%skip True
duplicate_rows = int(X.duplicated().sum())
print(f"Liczba zduplikowanych wierszy cech: {duplicate_rows}")


## 9. Statystyki opisowe

Podsumowanie pomaga szybko zauważyć zakresy wartości i różnice skali między cechami.


In [ ]:
# Sprawdź podstawowe statystyki opisowe dla każdej kolumny


In [ ]:
%%skip True
display(X.describe().T)


## 10. Nazwy i liczebność klas

Zamieniamy numery etykiet na czytelne nazwy gatunków i sprawdzamy rozkład klas.


In [ ]:
# w datasecie kategorie są numeryczne (0, 1, 2). Zamień to na kategorie tekstowe (setosa, versicolor, virginica). Nazwy te znajdziesz w iris.target_names
# policz ile jest przykładów dla każdej kategorii


In [ ]:
%%skip True
target_name_map = dict(enumerate(iris.target_names))
y_names = y.map(target_name_map)
class_counts = y_names.value_counts().sort_index()
display(class_counts.rename("count").to_frame())


## 11. Wykres liczebności klas

Tworzymy pierwszy artefakt graficzny i zapisujemy go jako plik PNG.


In [ ]:
# stwórz wykres kolumnowy pokazujący ile jest przykładów dla każdej kategorii


In [ ]:
%%skip True
ax = class_counts.plot.bar(
    title="Liczebność klas w zbiorze Iris",
    color="#4C78A8",
)
ax.set_xlabel("Klasa")
ax.set_ylabel("Liczba obserwacji")
ax.tick_params(axis="x", rotation=0)
ax.figure.tight_layout()
class_distribution_path = REPORTS_DIR / "class_distribution.png"
ax.figure.savefig(class_distribution_path, dpi=150)
plt.show()
plt.close(ax.figure)
print(f"Zapisano: {class_distribution_path}")


## 12–13. Funkcja walidująca dane

Funkcja realizuje podstawowy kontrakt danych: sprawdza obecność danych, zgodność liczby obserwacji, liczbę i typ cech, braki oraz oczekiwany zbiór klas. Każdy błąd ma czytelny komunikat.


In [ ]:
# stwórz funkcje walidującą dataset

def validate_data(features: pd.DataFrame, target: pd.Series) -> None:
    """Sprawdza podstawowy kontrakt zbioru Iris."""
    if features.empty:
        raise ValueError("Zbiór cech jest pusty.")
    if

In [ ]:
%%skip True
def validate_data(features: pd.DataFrame, target: pd.Series) -> None:
    """Sprawdza podstawowy kontrakt zbioru Iris."""
    if features.empty:
        raise ValueError("Zbiór cech jest pusty.")
    if len(features) != len(target):
        raise ValueError("Liczba wierszy cech i etykiet jest różna.")
    if features.shape[1] != 4:
        raise ValueError("Zbiór powinien zawierać dokładnie cztery cechy.")
    if features.isna().any().any():
        raise ValueError("Cechy zawierają brakujące wartości.")
    if not all(pd.api.types.is_numeric_dtype(dtype) for dtype in features.dtypes):
        raise TypeError("Wszystkie cechy powinny być numeryczne.")
    if set(target.unique()) != {0, 1, 2}:
        raise ValueError("Etykieta zawiera nieoczekiwane klasy.")


## 14. Test funkcji walidującej

Najpierw sprawdzamy poprawne dane. Następnie celowo usuwamy jedną cechę i potwierdzamy, że walidacja odrzuca błędny zbiór.


In [ ]:
validate_data(X, y)
print("Poprawne dane przeszły walidację.")

invalid_X = X.drop(columns=[X.columns[-1]])
try:
    validate_data(invalid_X, y)
except ValueError as error:
    print(f"Oczekiwany błąd dla niepoprawnych danych: {error}")
else:
    raise AssertionError("Walidacja powinna odrzucić dane bez jednej kolumny.")


## 15. Podział na dane treningowe i testowe

Wydzielamy 20% danych do końcowej ewaluacji. `stratify=y` zachowuje proporcje klas, a `random_state=42` umożliwia odtworzenie podziału.


In [ ]:
# dokończ

X_train, X_test, y_train, y_test = train_test_split(

In [ ]:
%%skip True
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"Trening: {X_train.shape}, test: {X_test.shape}")


## 16. Model bazowy

Klasyfikator bazowy (baseline) zawsze wybiera najczęstszą klasę. Stanowi minimalny punkt odniesienia dla właściwego modelu. Poprawnie powinno się zawsze tworzyć baseline i porównywać z nim czy nasz model działa lepiej. Np. dla klasyfikatora binarnego możemy stworzyć model który losowo zwraca 0 lub 1 z takim prawdopodobieństwem jak liczba klas (jeśli 0 stanowi 40% przypadków to 40% szans, że wynik będzie 0 a 60%, że 1) wtedy możemy zobaczyć czy nasz model jest lepszy niż losowanie próbek. Może się zdarzyć tak, że najbardziej złożone modele nie potrafią dawać lepszych wyników niż baseline bo np. nie ma odpowiednich informacji/zależności w danych. Za baseline można przyjąć też inny model. Np. jak wiemy, że model A działa dobrze ale chcemy stworzyć inny (lepszy) to towrzymy model B i porównujemy z modelem A.


In [ ]:
# Utwórzy prosty klasyfikator (DummyClassifier) który zwraca najczęściej występującą próbkę


In [ ]:
%%skip True
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)


## 17. Metryki modelu bazowego

Obliczamy accuracy i macro F1. Macro F1 nadaje każdej klasie taką samą wagę.


In [ ]:
# Użyj metryk do sprawdzenia skuteczności klasyfikatora
# użyk accuracy_score i f1_score


In [ ]:
%%skip True
baseline_accuracy = accuracy_score(y_test, baseline_predictions)
baseline_f1_macro = f1_score(
    y_test,
    baseline_predictions,
    average="macro",
)
print(f"Baseline accuracy: {baseline_accuracy:.4f}")
print(f"Baseline macro F1: {baseline_f1_macro:.4f}")


## 18. Potok modelu właściwego

Łączymy standaryzację i regresję logistyczną w jednym `Pipeline`. Dzięki temu ta sama transformacja zostanie zastosowana podczas treningu i predykcji.


In [ ]:
# utwórz Pipeline zawierający kroki StandardScaler i LogisticRegression
model = Pipeline(
    steps=[
        
    ]
)

In [ ]:
%%skip True
model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(max_iter=500, random_state=42),
        ),
    ]
)


## 19. Trening i predykcja

Dopasowujemy cały potok wyłącznie do danych treningowych, a następnie generujemy predykcje dla zbioru testowego.


In [ ]:
model.fit
predictions =

In [ ]:
%%skip True
model.fit(X_train, y_train)
predictions = model.predict(X_test)


## 20. Ewaluacja modelu właściwego

Obliczamy te same metryki co dla baseline oraz raport dla poszczególnych klas.


In [ ]:
# oblicz za pomocą accuracy_score, f1_score i classification_report

In [ ]:
%%skip True
model_accuracy = accuracy_score(y_test, predictions)
model_f1_macro = f1_score(y_test, predictions, average="macro")

print(f"Model accuracy: {model_accuracy:.4f}")
print(f"Model macro F1: {model_f1_macro:.4f}")
print()
print(
    classification_report(
        y_test,
        predictions,
        target_names=iris.target_names,
    )
)


## 21. Macierz pomyłek

Wykres pokazuje, które klasy są ze sobą mylone. Zapisujemy go jako drugi artefakt graficzny.


In [ ]:
# Utwórz macierz pomyłek za pomocą ConfusionMatrixDisplay i wyświetl używając matplotlib



In [ ]:
%%skip True
display_object = ConfusionMatrixDisplay.from_predictions(
    y_test,
    predictions,
    display_labels=iris.target_names,
    cmap="Blues",
)
display_object.ax_.set_title("Macierz pomyłek")
display_object.figure_.tight_layout()
confusion_matrix_path = REPORTS_DIR / "confusion_matrix.png"
display_object.figure_.savefig(confusion_matrix_path, dpi=150)
plt.show()
plt.close(display_object.figure_)
print(f"Zapisano: {confusion_matrix_path}")


## 22. Zapis metryk

Konwertujemy wartości na standardowy typ `float` i zapisujemy czytelny plik JSON.


In [ ]:
%%skip True
metrics = {
    "baseline_accuracy": float(baseline_accuracy),
    "baseline_f1_macro": float(baseline_f1_macro),
    "model_accuracy": float(model_accuracy),
    "model_f1_macro": float(model_f1_macro),
}

metrics_path = REPORTS_DIR / "metrics.json"
with metrics_path.open("w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=2)

print(json.dumps(metrics, indent=2))
print(f"Zapisano: {metrics_path}")


## 23. Zapis modelu i metadanych

W jednym artefakcie umieszczamy wytrenowany potok oraz informacje potrzebne do przygotowania prawidłowego wejścia i interpretacji wyjścia.


In [ ]:
model_artifact = {
    "model":
    "feature_names":
    "target_names":
}

joblib.dump()

In [ ]:
%%skip True
model_artifact = {
    "model": model,
    "feature_names": list(X.columns),
    "target_names": list(iris.target_names),
}
model_path = MODELS_DIR / "model.joblib"
joblib.dump(model_artifact, model_path)
print(f"Zapisano: {model_path}")


## 24. Weryfikacja artefaktów

Sprawdzamy automatycznie, czy każdy wymagany plik istnieje i nie jest pusty.


In [ ]:
required_files = [
    REPORTS_DIR / "class_distribution.png",
    REPORTS_DIR / "confusion_matrix.png",
    REPORTS_DIR / "metrics.json",
    MODELS_DIR / "model.joblib",
]

for file_path in required_files:
    if not file_path.exists():
        raise FileNotFoundError(f"Brak pliku: {file_path}")
    if file_path.stat().st_size == 0:
        raise ValueError(f"Plik jest pusty: {file_path}")
    print(f"OK: {file_path} ({file_path.stat().st_size} B)")
 

## 25. Podsumowanie

Regresja logistyczna osiąga wyższe macro F1 niż klasyfikator bazowy, dlatego jest lepszym z dwóch porównywanych modeli. Wynik należy interpretować razem z raportem klas i macierzą pomyłek, a nie tylko przez accuracy.

Notebook nie jest jeszcze systemem produkcyjnym. Brakuje między innymi wersjonowania danych i modelu, automatycznych testów, CI, kontraktu predykcji, wdrożenia oraz monitoringu.
